# Sentiment Analysis
**Project A - Group 10**

Notebook ini melakukan sentiment analysis menggunakan **TF-IDF + Logistic Regression** dan **TF-IDF + Naive Bayes** pada artikel yang sudah dipreprocessing.

Dataset: `scraped_articles_preprocessed.csv`
Label: `Positif`, `Negatif`, `Netral` (multi-class)

---

## 1. Install & Import Libraries

In [ ]:
!pip install scikit-learn pandas matplotlib seaborn -q

In [ ]:
import pandas as pd
import numpy as np
import ast
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    ConfusionMatrixDisplay
)
from sklearn.preprocessing import LabelEncoder

print("Libraries loaded successfully.")

## 2. Load Dataset

In [ ]:
# Load dataset langsung dari GitHub
DATA_URL = "https://raw.githubusercontent.com/MirzaFathi/Project-A---10/main/Data/scraped_articles_preprocessed.csv"

df = pd.read_csv(DATA_URL)
print(f"Shape: {df.shape}")
df.head(3)

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Distribusi label sentimen
print("Distribusi Sentimen:")
print(df['sentimen'].value_counts())
print()

# Distribusi kategori
print("Distribusi Kategori:")
print(df['category'].value_counts())

In [ ]:
# Visualisasi distribusi sentimen
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart sentimen
sentimen_counts = df['sentimen'].value_counts()
colors = ['#4CAF50', '#F44336', '#2196F3']
axes[0].bar(sentimen_counts.index, sentimen_counts.values, color=colors)
axes[0].set_title('Distribusi Label Sentimen')
axes[0].set_xlabel('Sentimen')
axes[0].set_ylabel('Jumlah Artikel')
for i, v in enumerate(sentimen_counts.values):
    axes[0].text(i, v + 2, str(v), ha='center', fontweight='bold')

# Bar chart kategori
cat_counts = df['category'].value_counts()
axes[1].barh(cat_counts.index, cat_counts.values, color='steelblue')
axes[1].set_title('Distribusi Kategori Artikel')
axes[1].set_xlabel('Jumlah Artikel')

plt.tight_layout()
plt.savefig('sentiment_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved.")

## 4. Persiapan Fitur

Kolom `stemming_cleaned` berisi hasil akhir preprocessing (tokenized, stopwords removed, stemmed).
Kita konversi list token tersebut menjadi string untuk TF-IDF.

In [ ]:
def tokens_to_string(val):
    """Konversi list token (string representasi) menjadi satu kalimat."""
    if isinstance(val, str):
        try:
            tokens = ast.literal_eval(val)
            if isinstance(tokens, list):
                return " ".join(tokens)
        except (ValueError, SyntaxError):
            return val
    return str(val)

df['text_features'] = df['stemming_cleaned'].apply(tokens_to_string)

# Cek hasilnya
print("Contoh text_features:")
print(df['text_features'].iloc[0][:200])
print()
print(f"Jumlah null: {df['text_features'].isnull().sum()}")

In [ ]:
# Label encoding
label_map = {'Positif': 2, 'Netral': 1, 'Negatif': 0}
df['label'] = df['sentimen'].map(label_map)

print("Label mapping:", label_map)
print(df[['sentimen', 'label']].drop_duplicates().sort_values('label'))

## 5. Train-Test Split

In [ ]:
X = df['text_features']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Training set : {len(X_train)} sampel")
print(f"Testing set  : {len(X_test)} sampel")
print()
print("Distribusi label di training set:")
print(y_train.map({v: k for k, v in label_map.items()}).value_counts())

## 6. TF-IDF Vectorization

TF-IDF (Term Frequency-Inverse Document Frequency) mengubah teks menjadi representasi numerik,
mempertimbangkan seberapa penting sebuah kata dalam dokumen relatif terhadap seluruh corpus.

In [ ]:
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),   # unigram + bigram
    min_df=2,
    max_df=0.95
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

print(f"Shape TF-IDF train : {X_train_tfidf.shape}")
print(f"Shape TF-IDF test  : {X_test_tfidf.shape}")

## 7. Model 1: Logistic Regression

Logistic Regression dipilih karena:
- Bekerja sangat baik dengan fitur TF-IDF (high-dimensional sparse)
- Mudah diinterpretasikan (koefisien per kata)
- Cepat dan stabil untuk multi-class classification

In [ ]:
lr_model = LogisticRegression(
    max_iter=1000,
    multi_class='multinomial',
    solver='lbfgs',
    C=1.0,
    random_state=42
)

lr_model.fit(X_train_tfidf, y_train)
y_pred_lr = lr_model.predict(X_test_tfidf)

acc_lr = accuracy_score(y_test, y_pred_lr)
print(f"Accuracy (Logistic Regression): {acc_lr:.4f} ({acc_lr*100:.2f}%)")

In [ ]:
# Classification report LR
label_names = ['Negatif', 'Netral', 'Positif']

print("=" * 55)
print("Classification Report - Logistic Regression")
print("=" * 55)
print(classification_report(y_test, y_pred_lr, target_names=label_names))

In [ ]:
# Confusion matrix LR
cm_lr = confusion_matrix(y_test, y_pred_lr)
disp_lr = ConfusionMatrixDisplay(confusion_matrix=cm_lr, display_labels=label_names)

fig, ax = plt.subplots(figsize=(6, 5))
disp_lr.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Confusion Matrix - Logistic Regression')
plt.tight_layout()
plt.savefig('cm_logistic_regression.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Model 2: Multinomial Naive Bayes

Naive Bayes dipilih sebagai model pembanding karena:
- Dirancang khusus untuk klasifikasi teks dengan data sparse
- Sangat efisien dan cepat
- Sering menjadi baseline yang kuat untuk sentiment analysis

In [ ]:
nb_model = MultinomialNB(alpha=0.1)
nb_model.fit(X_train_tfidf, y_train)
y_pred_nb = nb_model.predict(X_test_tfidf)

acc_nb = accuracy_score(y_test, y_pred_nb)
print(f"Accuracy (Naive Bayes): {acc_nb:.4f} ({acc_nb*100:.2f}%)")

In [ ]:
# Classification report NB
print("=" * 55)
print("Classification Report - Naive Bayes")
print("=" * 55)
print(classification_report(y_test, y_pred_nb, target_names=label_names))

In [ ]:
# Confusion matrix NB
cm_nb = confusion_matrix(y_test, y_pred_nb)
disp_nb = ConfusionMatrixDisplay(confusion_matrix=cm_nb, display_labels=label_names)

fig, ax = plt.subplots(figsize=(6, 5))
disp_nb.plot(ax=ax, colorbar=False, cmap='Oranges')
ax.set_title('Confusion Matrix - Naive Bayes')
plt.tight_layout()
plt.savefig('cm_naive_bayes.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Perbandingan Model

In [ ]:
from sklearn.metrics import f1_score

# Hitung metrik
metrics = {
    'Model': ['Logistic Regression', 'Naive Bayes'],
    'Accuracy': [acc_lr, acc_nb],
    'F1-Score (Macro)': [
        f1_score(y_test, y_pred_lr, average='macro'),
        f1_score(y_test, y_pred_nb, average='macro')
    ],
    'F1-Score (Weighted)': [
        f1_score(y_test, y_pred_lr, average='weighted'),
        f1_score(y_test, y_pred_nb, average='weighted')
    ]
}

df_metrics = pd.DataFrame(metrics).set_index('Model')
print(df_metrics.round(4))

In [ ]:
# Bar chart perbandingan
fig, ax = plt.subplots(figsize=(8, 4))

x = np.arange(len(df_metrics.columns))
width = 0.35

bars1 = ax.bar(x - width/2, df_metrics.loc['Logistic Regression'], width, label='Logistic Regression', color='steelblue')
bars2 = ax.bar(x + width/2, df_metrics.loc['Naive Bayes'], width, label='Naive Bayes', color='coral')

ax.set_ylabel('Score')
ax.set_title('Perbandingan Model: Logistic Regression vs Naive Bayes')
ax.set_xticks(x)
ax.set_xticklabels(df_metrics.columns)
ax.set_ylim(0, 1.05)
ax.legend()
ax.grid(axis='y', alpha=0.3)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Cross-Validation (5-Fold)

In [ ]:
from sklearn.pipeline import Pipeline

# Pipeline LR
pipe_lr = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1,2), min_df=2, max_df=0.95)),
    ('clf', LogisticRegression(max_iter=1000, multi_class='multinomial', solver='lbfgs', random_state=42))
])

# Pipeline NB
pipe_nb = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1,2), min_df=2, max_df=0.95)),
    ('clf', MultinomialNB(alpha=0.1))
])

cv_lr = cross_val_score(pipe_lr, X, y, cv=5, scoring='accuracy')
cv_nb = cross_val_score(pipe_nb, X, y, cv=5, scoring='accuracy')

print("Cross-Validation Accuracy (5-Fold):")
print(f"  Logistic Regression : {cv_lr.mean():.4f} +/- {cv_lr.std():.4f}")
print(f"  Naive Bayes         : {cv_nb.mean():.4f} +/- {cv_nb.std():.4f}")

## 11. Fitur Paling Berpengaruh per Kelas (Logistic Regression)

In [ ]:
feature_names = tfidf.get_feature_names_out()
top_n = 15

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
class_labels = ['Negatif', 'Netral', 'Positif']
bar_colors = ['#F44336', '#2196F3', '#4CAF50']

for i, (label, color) in enumerate(zip(class_labels, bar_colors)):
    coef = lr_model.coef_[i]
    top_indices = np.argsort(coef)[-top_n:]
    top_words = [feature_names[j] for j in top_indices]
    top_scores = [coef[j] for j in top_indices]

    axes[i].barh(top_words, top_scores, color=color, alpha=0.8)
    axes[i].set_title(f'Top {top_n} Kata - {label}')
    axes[i].set_xlabel('Coefficient')

plt.suptitle('Kata Paling Berpengaruh per Kelas Sentimen', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('top_features.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Prediksi Contoh Artikel

In [ ]:
reverse_label = {0: 'Negatif', 1: 'Netral', 2: 'Positif'}

# Ambil 5 sampel acak dari test set
sample_indices = np.random.RandomState(42).choice(len(X_test), size=5, replace=False)
X_sample = X_test.iloc[sample_indices]
y_sample_true = y_test.iloc[sample_indices]

X_sample_tfidf = tfidf.transform(X_sample)
y_sample_pred = lr_model.predict(X_sample_tfidf)

print(f"{'No':<4} {'Actual':<12} {'Predicted':<12} {'Match'}")
print("-" * 42)
for i, (true, pred) in enumerate(zip(y_sample_true, y_sample_pred)):
    match = 'v' if true == pred else 'x'
    print(f"{i+1:<4} {reverse_label[true]:<12} {reverse_label[pred]:<12} {match}")

## 13. Kesimpulan

| Model | Kelebihan | Kekurangan |
|---|---|---|
| **Logistic Regression** | Akurasi lebih tinggi, interpretable, stabil | Lebih lambat dari NB |
| **Naive Bayes** | Sangat cepat, bagus sebagai baseline | Asumsi independen tidak realistis |

**Model yang direkomendasikan: Logistic Regression**, karena:
- Umumnya menghasilkan akurasi dan F1-score lebih tinggi pada data TF-IDF
- Cocok untuk dataset berukuran sedang (804 artikel)
- Hasil cross-validation lebih stabil

Model ini memprediksi sentimen artikel (Positif / Netral / Negatif) berdasarkan fitur TF-IDF dari teks yang sudah dipreprocessing.